In [38]:
import pandas as pd
!%pip install pyreadr
import pyreadr



'%pip' is not recognized as an internal or external command,
operable program or batch file.


In [40]:
df_lookup = pyreadr.read_r(r"\\main.oecd.org\ASgenDCD\DEVELOPMENT_AID\__PROCESSING\checklist_r\3_share\lookups_checklist.RData")
df_lookup.keys()

odict_keys(['annex2_lookup', 'channels_list', 'currency_list', 'donor_agency_lookup', 'donor_countries', 'finance_list', 'list_guarantees', 'long_channels_list', 'lookup_donor', 'multilateral_donors', 'parent_channel_lookup', 'private_donors', 'purposes_list', 'recipients_list', 'regional_recipients'])

In [3]:
df = pd.read_excel(r"\\main.oecd.org\ASgenDCD\DEVELOPMENT_AID\__PROCESSING\checklist_r\2_functions\test_crstemplate_GG.xlsx",header = 1)
df = df.loc[1:].reset_index(drop=True)
df["error_message"] = ""
df["warning_message"] = ""


In [4]:
df.head()  # Display the first few rows of the DataFrame


,Reporting year,Commitment date (dd-mm-yyyy),Reporting country / organisation,Extending agency,CRS Identification N°,Donor project N°,Nature of submission,Recipient code,Channel of delivery name,Channel code,...,Average use of portfolio guarantee %,PSI flag,Additionality type,Additionality assessment,Additionality – development objective,Income Group,Discount Rate,Grant Element,error_message,warning_message
0,2024,2022-01-15 00:00:00,1,1,20220000001,abcd,1,998,African Development Bank,12004.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,
1,2024,2024-01-15 00:00:00,1,1,20220000002,abcd,1,249,International Committee of the Red Cross,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,
2,2024,2021-12-31 00:00:00,1,1,20220000003,abcd,3,998,Institute for Advanced Research and Policy Stu...,51010.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,
3,2024,2024-01-01 00:00:00,1,1,20220000004,abcd,1,998,Republic of Ireland,12003.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,
4,2024,2024-01-01 00:00:00,7000,1,20220000005,abcd,8,998,Private sector partnership with local small an...,21045.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,


In [5]:
df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
        (df["Type of flow (main DAC 1 category)"]== 21 ) | 
          (df["Type of flow (main DAC 1 category)"]== 60 )) &
            (df["Co-operation modality (replaces the type of aid)"]== "A01") &
              ((~df["Sector / Purpose code and corresponding shares"].astype(str).str.contains("51010", na=False) | 
                (~df["Sector / Purpose code and corresponding shares"].astype(str).str.contains("51010:100", na=False)))), "warning_message"] = df["warning_message"] + "PPC should be 51010 when the Co-operation modality is A01"

In [6]:
df[["Type of flow (main DAC 1 category)","Co-operation modality (replaces the type of aid)",  "Sector / Purpose code and corresponding shares", "warning_message"]].head(30)

,Type of flow (main DAC 1 category),Co-operation modality (replaces the type of aid),Sector / Purpose code and corresponding shares,warning_message
0,10,NaN,51010:20.12|45866:30|99810:40,
1,10,A02,99810,
2,10,A02,42559;12264,
3,21,NaN,41000,
4,10,B01,23231,
5,10,B01,45866,
6,60,NaN,43060,
7,10,B01,51010:19.5|45866:30|51010:50.5,
8,10,B01,12240,
9,10,A01,12240:100,PPC should be 51010 when the Co-operation moda...


In [7]:
df["Co-operation modality (replaces the type of aid)"].isna().sum()  # Count of missing values in the specified column

4

In [8]:
df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
        (df["Type of flow (main DAC 1 category)"]== 21 ) | 
          (df["Type of flow (main DAC 1 category)"]== 60 )) &
            (df["Co-operation modality (replaces the type of aid)"]== "A01") &
              (df["Channel code"].astype(str).str[:2]!="12"), "error_message"] = df["error_message"] + "Parent channel should be 12000 when the Co-operation modality is A01"


In [9]:
df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
        (df["Type of flow (main DAC 1 category)"]== 21 ) | 
          (df["Type of flow (main DAC 1 category)"]== 60 )) 
          & (df["Co-operation modality (replaces the type of aid)"]== "A01") 
          & (df["Channel code"].astype(str).str[:2]!="12")]


,Reporting year,Commitment date (dd-mm-yyyy),Reporting country / organisation,Extending agency,CRS Identification N°,Donor project N°,Nature of submission,Recipient code,Channel of delivery name,Channel code,...,Average use of portfolio guarantee %,PSI flag,Additionality type,Additionality assessment,Additionality – development objective,Income Group,Discount Rate,Grant Element,error_message,warning_message
9,2024,2022-01-15 00:00:00,1,1,20220000002,abcd,1,249,International Committee of the Red Cross,21009.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Parent channel should be 12000 when the Co-ope...,PPC should be 51010 when the Co-operation moda...
10,2024,2022-01-15 00:00:00,1,1,20220000002,abcd,1,3000,International Committee of the Red Cross,21009.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Parent channel should be 12000 when the Co-ope...,PPC should be 51010 when the Co-operation moda...


In [10]:
df[["Type of flow (main DAC 1 category)","Sector / Purpose code and corresponding shares", "Co-operation modality (replaces the type of aid)", "error_message"]].head(20)


,Type of flow (main DAC 1 category),Sector / Purpose code and corresponding shares,Co-operation modality (replaces the type of aid),error_message
0,10,51010:20.12|45866:30|99810:40,NaN,
1,10,99810,A02,
2,10,42559;12264,A02,
3,21,41000,NaN,
4,10,23231,B01,
5,10,45866,B01,
6,60,43060,NaN,
7,10,51010:19.5|45866:30|51010:50.5,B01,
8,10,12240,B01,
9,10,12240:100,A01,Parent channel should be 12000 when the Co-ope...


In [11]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [12]:
df.loc[
    (df["Co-operation modality (replaces the type of aid)"]== "A02") &
            ((df["Sector / Purpose code and corresponding shares"].str.contains("99810", na=False)) | 
                (df["Sector / Purpose code and corresponding shares"].str.contains("99810:100", na=False)))]

,Reporting year,Commitment date (dd-mm-yyyy),Reporting country / organisation,Extending agency,CRS Identification N°,Donor project N°,Nature of submission,Recipient code,Channel of delivery name,Channel code,Bi/Multi,Type of flow (main DAC 1 category),Type of finance,Co-operation modality (replaces the type of aid),Short description / Project title,Sector / Purpose code and corresponding shares,Geographical target area,Regional aid to LDCs (codes 1-5),Expected starting date,Expected completion date,Description,SDG focus*,Keywords*,Gender equality,Aid to environment,DIG,RMNCH,Disaster Risk Reduction,Nutrition*,Inclusion and empowerment of persons with disabilities*,FTC,PBA,Investment,Type of blended finance (codes 1-4),Biodiversity,Climate change - mitigation,Climate change - adaptation,Desertification,Currency,Commitments,Capital Expenditure %*,Amounts extended,ODA grant equivalent,Amounts received (for loans:principal only),Amount untied,Amount partially untied,Amount tied,Amount of IRTC,"If project type, amount of experts-commitments*","If project type, amount of experts-extended*",Amount of export credit,Leveraging mechanism and role/position,Amounts mobilised from the private sector,Origin of the funds mobilised,Type or repayment \nor type of fee payment,Number of repayment \nor fee payment per annum,Interest rate /\n Fee rate /\nExpected return per annum,Second interest rate,First repayment date /\nExposure reduction starting date \n(dd-mm-yyyy),Final repayment date / \nGuarantee maturity date / \nExpected maturity \n(dd-mm-yyyy),Interest received / \nGuarantee fee received / \nDividends received per annum,Principal disbursed and still outstanding / \nEquity disbursed and still held,Arrears of principal (included in the item 62),Arrears of interest / \nArrears of guarantee fee,Guaranteed amount,Average use of portfolio guarantee %,PSI flag,Additionality type,Additionality assessment,Additionality – development objective,Income Group,Discount Rate,Grant Element,error_message,warning_message


In [13]:
df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
        (df["Type of flow (main DAC 1 category)"]== 21 ) | 
          (df["Type of flow (main DAC 1 category)"]== 60 )) &
            (df["Co-operation modality (replaces the type of aid)"]== "A02") &
              ( (df["Sector / Purpose code and corresponding shares"].astype(str).str.contains("99810", na=False)) | 
                (df["Sector / Purpose code and corresponding shares"].astype(str).str.contains("99810:100", na=False)))]

,Reporting year,Commitment date (dd-mm-yyyy),Reporting country / organisation,Extending agency,CRS Identification N°,Donor project N°,Nature of submission,Recipient code,Channel of delivery name,Channel code,Bi/Multi,Type of flow (main DAC 1 category),Type of finance,Co-operation modality (replaces the type of aid),Short description / Project title,Sector / Purpose code and corresponding shares,Geographical target area,Regional aid to LDCs (codes 1-5),Expected starting date,Expected completion date,Description,SDG focus*,Keywords*,Gender equality,Aid to environment,DIG,RMNCH,Disaster Risk Reduction,Nutrition*,Inclusion and empowerment of persons with disabilities*,FTC,PBA,Investment,Type of blended finance (codes 1-4),Biodiversity,Climate change - mitigation,Climate change - adaptation,Desertification,Currency,Commitments,Capital Expenditure %*,Amounts extended,ODA grant equivalent,Amounts received (for loans:principal only),Amount untied,Amount partially untied,Amount tied,Amount of IRTC,"If project type, amount of experts-commitments*","If project type, amount of experts-extended*",Amount of export credit,Leveraging mechanism and role/position,Amounts mobilised from the private sector,Origin of the funds mobilised,Type or repayment \nor type of fee payment,Number of repayment \nor fee payment per annum,Interest rate /\n Fee rate /\nExpected return per annum,Second interest rate,First repayment date /\nExposure reduction starting date \n(dd-mm-yyyy),Final repayment date / \nGuarantee maturity date / \nExpected maturity \n(dd-mm-yyyy),Interest received / \nGuarantee fee received / \nDividends received per annum,Principal disbursed and still outstanding / \nEquity disbursed and still held,Arrears of principal (included in the item 62),Arrears of interest / \nArrears of guarantee fee,Guaranteed amount,Average use of portfolio guarantee %,PSI flag,Additionality type,Additionality assessment,Additionality – development objective,Income Group,Discount Rate,Grant Element,error_message,warning_message
1,2024,2024-01-15 00:00:00,1,1,20220000002,abcd,1,249,International Committee of the Red Cross,NaN,8,10,110,A02,Programme d'agriculture durable et de sécurité...,99810,"The Mekong Delta region in Vietnam, which is a...",3.0,NaN,NaN,This project seeks to promote sustainable agri...,1.2|1.4,NaN,2,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,2.0,2.0,2,2,75,NaN,NaN,10000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,


In [14]:
df[(df["Co-operation modality (replaces the type of aid)"]=="A02")&
             (df["Sector / Purpose code and corresponding shares"].astype(str).str.contains("99810", na=False))]

,Reporting year,Commitment date (dd-mm-yyyy),Reporting country / organisation,Extending agency,CRS Identification N°,Donor project N°,Nature of submission,Recipient code,Channel of delivery name,Channel code,Bi/Multi,Type of flow (main DAC 1 category),Type of finance,Co-operation modality (replaces the type of aid),Short description / Project title,Sector / Purpose code and corresponding shares,Geographical target area,Regional aid to LDCs (codes 1-5),Expected starting date,Expected completion date,Description,SDG focus*,Keywords*,Gender equality,Aid to environment,DIG,RMNCH,Disaster Risk Reduction,Nutrition*,Inclusion and empowerment of persons with disabilities*,FTC,PBA,Investment,Type of blended finance (codes 1-4),Biodiversity,Climate change - mitigation,Climate change - adaptation,Desertification,Currency,Commitments,Capital Expenditure %*,Amounts extended,ODA grant equivalent,Amounts received (for loans:principal only),Amount untied,Amount partially untied,Amount tied,Amount of IRTC,"If project type, amount of experts-commitments*","If project type, amount of experts-extended*",Amount of export credit,Leveraging mechanism and role/position,Amounts mobilised from the private sector,Origin of the funds mobilised,Type or repayment \nor type of fee payment,Number of repayment \nor fee payment per annum,Interest rate /\n Fee rate /\nExpected return per annum,Second interest rate,First repayment date /\nExposure reduction starting date \n(dd-mm-yyyy),Final repayment date / \nGuarantee maturity date / \nExpected maturity \n(dd-mm-yyyy),Interest received / \nGuarantee fee received / \nDividends received per annum,Principal disbursed and still outstanding / \nEquity disbursed and still held,Arrears of principal (included in the item 62),Arrears of interest / \nArrears of guarantee fee,Guaranteed amount,Average use of portfolio guarantee %,PSI flag,Additionality type,Additionality assessment,Additionality – development objective,Income Group,Discount Rate,Grant Element,error_message,warning_message
1,2024,2024-01-15 00:00:00,1,1,20220000002,abcd,1,249,International Committee of the Red Cross,NaN,8,10,110,A02,Programme d'agriculture durable et de sécurité...,99810,"The Mekong Delta region in Vietnam, which is a...",3.0,NaN,NaN,This project seeks to promote sustainable agri...,1.2|1.4,NaN,2,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,2.0,2.0,2,2,75,NaN,NaN,10000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,


In [15]:
df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
        (df["Type of flow (main DAC 1 category)"]== 21 ) | 
          (df["Type of flow (main DAC 1 category)"]== 60 )) &
             (df["Co-operation modality (replaces the type of aid)"]== "A02") &
             ((df["Sector / Purpose code and corresponding shares"].astype(str).str.contains("51010", na=False))| 
              (df["Sector / Purpose code and corresponding shares"].astype(str).str.contains("51010:100", na=False)) | 
              (df["Sector / Purpose code and corresponding shares"].astype(str).str.contains("99810", na=False)) | 
              (df["Sector / Purpose code and corresponding shares"].astype(str).str.contains("99810:100", na=False)))]

,Reporting year,Commitment date (dd-mm-yyyy),Reporting country / organisation,Extending agency,CRS Identification N°,Donor project N°,Nature of submission,Recipient code,Channel of delivery name,Channel code,Bi/Multi,Type of flow (main DAC 1 category),Type of finance,Co-operation modality (replaces the type of aid),Short description / Project title,Sector / Purpose code and corresponding shares,Geographical target area,Regional aid to LDCs (codes 1-5),Expected starting date,Expected completion date,Description,SDG focus*,Keywords*,Gender equality,Aid to environment,DIG,RMNCH,Disaster Risk Reduction,Nutrition*,Inclusion and empowerment of persons with disabilities*,FTC,PBA,Investment,Type of blended finance (codes 1-4),Biodiversity,Climate change - mitigation,Climate change - adaptation,Desertification,Currency,Commitments,Capital Expenditure %*,Amounts extended,ODA grant equivalent,Amounts received (for loans:principal only),Amount untied,Amount partially untied,Amount tied,Amount of IRTC,"If project type, amount of experts-commitments*","If project type, amount of experts-extended*",Amount of export credit,Leveraging mechanism and role/position,Amounts mobilised from the private sector,Origin of the funds mobilised,Type or repayment \nor type of fee payment,Number of repayment \nor fee payment per annum,Interest rate /\n Fee rate /\nExpected return per annum,Second interest rate,First repayment date /\nExposure reduction starting date \n(dd-mm-yyyy),Final repayment date / \nGuarantee maturity date / \nExpected maturity \n(dd-mm-yyyy),Interest received / \nGuarantee fee received / \nDividends received per annum,Principal disbursed and still outstanding / \nEquity disbursed and still held,Arrears of principal (included in the item 62),Arrears of interest / \nArrears of guarantee fee,Guaranteed amount,Average use of portfolio guarantee %,PSI flag,Additionality type,Additionality assessment,Additionality – development objective,Income Group,Discount Rate,Grant Element,error_message,warning_message
1,2024,2024-01-15 00:00:00,1,1,20220000002,abcd,1,249,International Committee of the Red Cross,NaN,8,10,110,A02,Programme d'agriculture durable et de sécurité...,99810,"The Mekong Delta region in Vietnam, which is a...",3.0,NaN,NaN,This project seeks to promote sustainable agri...,1.2|1.4,NaN,2,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,2.0,2.0,2,2,75,NaN,NaN,10000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,


In [16]:
df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
        (df["Type of flow (main DAC 1 category)"]== 21 ) | 
          (df["Type of flow (main DAC 1 category)"]== 60 )) &
             (df["Co-operation modality (replaces the type of aid)"]== "A02") &
             ((df["Sector / Purpose code and corresponding shares"].astype(str).str.contains("51010", na=False))| 
              (df["Sector / Purpose code and corresponding shares"].astype(str).str.contains("51010:100", na=False)) | 
              (df["Sector / Purpose code and corresponding shares"].astype(str).str.contains("99810", na=False)) | 
              (df["Sector / Purpose code and corresponding shares"].astype(str).str.contains("99810:100", na=False))), "error_message"] = "PPC should be different from 51010 or 99810 when the Co-operation modality is A01"

In [17]:
df[["Type of flow (main DAC 1 category)","Sector / Purpose code and corresponding shares", "Co-operation modality (replaces the type of aid)", "error_message"]].head(20)


,Type of flow (main DAC 1 category),Sector / Purpose code and corresponding shares,Co-operation modality (replaces the type of aid),error_message
0,10,51010:20.12|45866:30|99810:40,NaN,
1,10,99810,A02,PPC should be different from 51010 or 99810 wh...
2,10,42559;12264,A02,
3,21,41000,NaN,
4,10,23231,B01,
5,10,45866,B01,
6,60,43060,NaN,
7,10,51010:19.5|45866:30|51010:50.5,B01,
8,10,12240,B01,
9,10,12240:100,A01,Parent channel should be 12000 when the Co-ope...


In [18]:
df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
        (df["Type of flow (main DAC 1 category)"]== 21 ) | 
          (df["Type of flow (main DAC 1 category)"]== 60 )) &
            (df["Co-operation modality (replaces the type of aid)"]== "A02") &
              (df["Channel code"].astype(str).str[:2]!="12"), "error_message"] = df["error_message"] + " ;Parent channel should be 12000 when the Co-operation modality is A02"


In [19]:
print(df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
        (df["Type of flow (main DAC 1 category)"]== 21 ) | 
          (df["Type of flow (main DAC 1 category)"]== 60 )) &
            (df["Co-operation modality (replaces the type of aid)"]== "A02") &
              (df["Channel code"].astype(str).str[:2]!="12")][["Co-operation modality (replaces the type of aid)","Channel code","error_message"]])

  Co-operation modality (replaces the type of aid)  Channel code                                      error_message
1                                              A02           NaN  PPC should be different from 51010 or 99810 wh...
2                                              A02       51010.0   ;Parent channel should be 12000 when the Co-o...


In [20]:
df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
        (df["Type of flow (main DAC 1 category)"]== 21 ) | 
          (df["Type of flow (main DAC 1 category)"]== 60 )) &
            (df["Co-operation modality (replaces the type of aid)"]== "A02") &
              (df["Channel code"].astype(str).str[:2]!="12")]["error_message"][0:1].item()

'PPC should be different from 51010 or 99810 when the Co-operation modality is A01 ;Parent channel should be 12000 when the Co-operation modality is A02'

In [21]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [22]:
df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
        (df["Type of flow (main DAC 1 category)"]== 21 ) | 
          (df["Type of flow (main DAC 1 category)"]== 60 )) &
            (((df["Co-operation modality (replaces the type of aid)"]== "B01") &
              (~df["Bi/Multi"].isin([3,7])))| ((df["Co-operation modality (replaces the type of aid)"]!= "B01") &
              (df["Bi/Multi"].isin([3,7])))), "error_message"] = df["error_message"] + " ;Bi_multi should be 3 or 7 when co-operation modality is B01, and vice versa"

In [23]:
df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
        (df["Type of flow (main DAC 1 category)"]== 21 ) | 
          (df["Type of flow (main DAC 1 category)"]== 60 )) &
            (df["Co-operation modality (replaces the type of aid)"]== "B01") &
              (~df["Channel code"].astype(str).str[:2].isin(["20", "21","30","51"])), "error_message"] = df["error_message"] + " ;Parent channel should be 20000, 30000 or 51000 when co-operation modality is B01"


In [24]:
#df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
 #       (df["Type of flow (main DAC 1 category)"]== 21 ) | 
  #        (df["Type of flow (main DAC 1 category)"]== 60 )) &
   #         (df["Co-operation modality (replaces the type of aid)"]== "B02") , "warning_message"] = df["warning_message"] + " ;Please verify if it is possible to assign a more specific co-operation modality (e.g. B021, B022)"


In [25]:
df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
        (df["Type of flow (main DAC 1 category)"]== 21 ) | 
          (df["Type of flow (main DAC 1 category)"]== 60 )) &
            (df["Co-operation modality (replaces the type of aid)"].astype(str).str.startswith("B02")) &
              (df["Bi/Multi"] != 2), "error_message"] = df["error_message"] + " ;Bi_multi should be 2 when co-operation modality is B02,B021 or B022 and vice versa"


In [26]:
df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
        (df["Type of flow (main DAC 1 category)"]== 21 ) | 
          (df["Type of flow (main DAC 1 category)"]== 60 )) &
            (((df["Co-operation modality (replaces the type of aid)"].astype(str).str.startswith("B02")) &
              (df["Recipient code"] != 3000)) | ((~df["Co-operation modality (replaces the type of aid)"].astype(str).str.startswith("B02")) &
              (df["Recipient code"] == 3000))), "error_message"] = df["error_message"] + " ;The recipient code for core contributions should be 3000"


In [27]:
#df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
 #       (df["Type of flow (main DAC 1 category)"]== 21 ) | 
  #        (df["Type of flow (main DAC 1 category)"]== 60 )) &
   #         (df["Co-operation modality (replaces the type of aid)"]== "B03") , "warning_message"] = df["warning_message"] + " ;Please verify if it is possible to assign a more specific co-operation modality (e.g. B031, B032, B033)"

In [ ]:
df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
        (df["Type of flow (main DAC 1 category)"]== 21 ) | 
          (df["Type of flow (main DAC 1 category)"]== 60 )) &
            (df["Co-operation modality (replaces the type of aid)"].astype(str).str.startswith("B03")) &
              (~df["Bi/Multi"].isin([1,8])), "error_message"] = df["error_message"] + " ;Bi-multi should be 1 (or in some cases 8) when co-operation modality is B03x"


In [30]:
df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
        (df["Type of flow (main DAC 1 category)"]== 21 ) | 
          (df["Type of flow (main DAC 1 category)"]== 60 )) &
            (df["Co-operation modality (replaces the type of aid)"].astype(str).str.startswith("B03")) &
              (~df["Channel code"].astype(str).str[:2].isin(["20", "21","30","40","51"])), "error_message"] = df["error_message"] + " ;If co-operation modality = B03x, channel should be in parent channel category 20000, 30000, 40000, or 51000"


In [ ]:
df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
       (df["Type of flow (main DAC 1 category)"]== 21 ) | 
        (df["Type of flow (main DAC 1 category)"]== 60 )) &
         (df["Co-operation modality (replaces the type of aid)"]!= "C01")&
         (df["Investment"==1]) , "warning_message"] = df["warning_message"] + " ;Generally, co-operation modality should = C01 when investment = 1"

In [ ]:
#ADD F01 checks

In [31]:
df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
        (df["Type of flow (main DAC 1 category)"]== 21 ) | 
          (df["Type of flow (main DAC 1 category)"]== 60 )) &
            (df["Co-operation modality (replaces the type of aid)"]== "G01") &
              ((~df["Sector / Purpose code and corresponding shares"].astype(str).str.contains("51010", na=False) | 
                (~df["Sector / Purpose code and corresponding shares"].astype(str).str.contains("51010:100", na=False)))), "warning_message"] = df["warning_message"] + " ;Generally, purpose code is 91010 when co-operation modality is G01"

In [32]:
df.loc[((df["Type of flow (main DAC 1 category)"]== 10) |
        (df["Type of flow (main DAC 1 category)"]== 21 ) | 
          (df["Type of flow (main DAC 1 category)"]== 60 )) &
            (df["Co-operation modality (replaces the type of aid)"]!= "G01") &
              ((df["Sector / Purpose code and corresponding shares"].astype(str).str.contains("51010", na=False) | 
                (df["Sector / Purpose code and corresponding shares"].astype(str).str.contains("51010:100", na=False)))), "error_message"] = df["error_message"] + " ;Co-operation modality should be G01 when purpose code is 91010"

In [ ]:
df.loc[
    (df["Co-operation modality (replaces the type of aid)"]== "G01") &
            ((df["Sector / Purpose code and corresponding shares"].str.contains("99810", na=False)) | 
                (df["Sector / Purpose code and corresponding shares"].str.contains("99810:100", na=False))), "error_message"] = df["error_message"] + " ;Purpose code 99810 is not accepted when co-operation modality is G01"